# Лабораторна робота №8

**Тема:** Створення простої програми машинного навчання на Python  
**Мета:** Набути навичок використання бібліотек `NumPy`, `SciPy` та `Matplotlib`.

У ноутбуці виконано:
- завантаження та очищення даних `web_traffic.tsv`;
- побудову графіка трафіку;
- апроксимацію поліномами степенів `1`, `2`, `3`, `20`, `40`;
- аналіз точки перегину;
- train/test перевірку моделей;
- прогноз моменту досягнення `100000 hits/hour`.


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

warnings.simplefilter("ignore", np.RankWarning)

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["grid.alpha"] = 0.5

SEED = 42
rng = np.random.default_rng(SEED)


## 1. Завантаження даних

Файл `web_traffic.tsv` містить дві колонки:
- `x` — номер години;
- `y` — кількість запитів за цю годину.


In [ ]:
data = np.genfromtxt("web_traffic.tsv", delimiter="\t")

print("Перші 10 рядків даних:")
print(data[:10])
print("\nФорма масиву:", data.shape)


In [ ]:
x = data[:, 0]
y = data[:, 1]

nan_count = np.sum(np.isnan(y))
print("Кількість значень NaN:", nan_count)

x = x[~np.isnan(y)]
y = y[~np.isnan(y)]

print("Розмір очищених даних:", x.shape, y.shape)


## 2. Візуалізація даних

Побудуємо діаграму розсіювання та подивимося, як змінюється трафік у часі.


In [ ]:
plt.scatter(x, y, s=10, color="tab:blue")
plt.title("Web traffic over the last month")
plt.xlabel("Time")
plt.ylabel("Hits/hour")
plt.xticks(
    [w * 7 * 24 for w in range(11)],
    [f"week {w}" for w in range(11)]
)
plt.autoscale(tight=True)
plt.show()


## 3. Побудова моделей

Побудуємо поліноміальні моделі та обчислимо похибку апроксимації.


In [ ]:
def error(f, x, y):
    return np.sum((f(x) - y) ** 2)


def fit_model(x, y, degree):
    coeffs = np.polyfit(x, y, degree)
    model = np.poly1d(coeffs)
    return coeffs, model


def polynomial_to_string(model):
    return str(model).replace("\n", " ")


fx = np.linspace(0, x[-1], 1000)


In [ ]:
f1_coeffs, f1 = fit_model(x, y, 1)
f1_error = error(f1, x, y)

print("Параметри моделі degree=1:", f1_coeffs)
print("Рівняння прямої:")
print(f1)
print("Похибка апроксимації:", f1_error)

plt.scatter(x, y, s=10, label="data")
plt.plot(fx, f1(fx), linewidth=3, color="tab:red", label="d=1")
plt.title("Лінійна апроксимація")
plt.xlabel("Time")
plt.ylabel("Hits/hour")
plt.xticks(
    [w * 7 * 24 for w in range(11)],
    [f"week {w}" for w in range(11)]
)
plt.legend(loc="upper left")
plt.show()


In [ ]:
degrees = [1, 2, 3, 20, 40]
models = {}
rows = []

for degree in degrees:
    coeffs, model = fit_model(x, y, degree)
    current_error = error(model, x, y)
    models[degree] = model
    rows.append({
        "degree": degree,
        "approx_error": current_error
    })
    print(f"degree={degree}, error={current_error:.2f}")

errors_df = pd.DataFrame(rows).sort_values("degree")
errors_df


In [ ]:
plt.scatter(x, y, s=10, label="data")

colors = {
    1: "tab:red",
    2: "tab:green",
    3: "tab:orange",
    20: "tab:purple",
    40: "tab:brown",
}

for degree in degrees:
    plt.plot(fx, models[degree](fx), linewidth=2, color=colors[degree], label=f"d={degree}")

plt.title("Порівняння поліноміальних моделей")
plt.xlabel("Time")
plt.ylabel("Hits/hour")
plt.xticks(
    [w * 7 * 24 for w in range(11)],
    [f"week {w}" for w in range(11)]
)
plt.ylim(bottom=0)
plt.legend(loc="upper left")
plt.show()


## 4. Точка перегину

Розділимо дані в точці `3.5` тижні та навчимо дві окремі прямі.


In [ ]:
inflection = int(3.5 * 7 * 24)

xa = x[x < inflection]
ya = y[x < inflection]
xb = x[x >= inflection]
yb = y[x >= inflection]

fa = np.poly1d(np.polyfit(xa, ya, 1))
fb = np.poly1d(np.polyfit(xb, yb, 1))

fa_error = error(fa, xa, ya)
fb_error = error(fb, xb, yb)
piecewise_error = fa_error + fb_error

print(f"Error inflection = {piecewise_error:.2f}")

plt.scatter(x, y, s=10, label="data")
plt.plot(xa, fa(xa), linewidth=3, color="tab:red", label="before inflection")
plt.plot(xb, fb(xb), linewidth=3, color="tab:green", label="after inflection")
plt.axvline(inflection, color="black", linestyle="--", label="3.5 weeks")
plt.title("Апроксимація двома прямими")
plt.xlabel("Time")
plt.ylabel("Hits/hour")
plt.xticks(
    [w * 7 * 24 for w in range(11)],
    [f"week {w}" for w in range(11)]
)
plt.legend(loc="upper left")
plt.show()


## 5. Відкладена вибірка (train/test)

Оцінимо моделі не тільки за помилкою на всіх даних, а й за помилкою на тестовій вибірці. Це дає більш реалістичну оцінку узагальнення.


In [ ]:
indices = np.arange(len(xb))
rng.shuffle(indices)

split_point = int(len(indices) * 0.7)
train_idx = indices[:split_point]
test_idx = indices[split_point:]

x_train, y_train = xb[train_idx], yb[train_idx]
x_test, y_test = xb[test_idx], yb[test_idx]

test_rows = []
winner_degree = None
winner_model = None
best_test_error = float("inf")

for degree in degrees:
    model = np.poly1d(np.polyfit(x_train, y_train, degree))
    train_error = error(model, x_train, y_train)
    test_error = error(model, x_test, y_test)
    test_rows.append({
        "degree": degree,
        "train_error": train_error,
        "test_error": test_error
    })

    if test_error < best_test_error:
        best_test_error = test_error
        winner_degree = degree
        winner_model = model

test_results_df = pd.DataFrame(test_rows).sort_values("degree")
print("Найкраща модель за тестовою похибкою: degree =", winner_degree)
test_results_df


In [ ]:
future_x = np.linspace(inflection, 1000, 500)

plt.scatter(xb, yb, s=10, label="data after inflection")
plt.scatter(x_train, y_train, s=18, label="train", color="tab:green")
plt.scatter(x_test, y_test, s=18, label="test", color="tab:orange")
plt.plot(future_x, winner_model(future_x), linewidth=3, color="tab:red", label=f"winner d={winner_degree}")
plt.title("Найкраща модель на відкладених даних")
plt.xlabel("Time")
plt.ylabel("Hits/hour")
plt.xticks(
    [w * 7 * 24 for w in range(11)],
    [f"week {w}" for w in range(11)]
)
plt.legend(loc="upper left")
plt.show()


## 6. Прогноз досягнення `100000 hits/hour`

Знайдемо момент часу, коли найкраща модель досягне `100000` запитів за годину.


In [ ]:
print("Winner model:")
print(winner_model)

reached_max = fsolve(winner_model - 100000, x0=800)[0]
reached_week = reached_max / (7 * 24)

print(f"100000 hits/hour expected at hour {reached_max:.2f}")
print(f"100000 hits/hour expected at week {reached_week:.2f}")
